# Feature Temporal-Availability Audit

Bitcoin Transaction Forensics -- audit of the 166 inherited Elliptic++ transaction features
(`Time step` + `Local_feature_1..93` + `Aggregate_feature_1..72`).

**Goal:** for each inherited feature, determine whether there is evidence it depends on
wallet/address information that is itself static full-history rather than point-in-time.

Two separate questions per feature, never conflated:
- **Question A:** can the feature be substantially reconstructed from wallet-level variables?
- **Question B:** are those wallet-level variables themselves static full-history (not point-in-time)?

A feature only earns a "strong temporal-availability concern" label when there's evidence for **both**.

This is a separate notebook from `02_Data_Processing.ipynb` (which holds the official Model 1/2
baselines and the wallet-reconstruction findings that motivated this audit) so this staged,
compute-aware investigation doesn't keep growing that notebook indefinitely. Findings here build
on, and do not replace, what's already established there.

## Setup: reproduce the labelled dataset and a reference RF model

**Reproducibility fix applied here:** `02_Data_Processing.ipynb` sorts the labelled data with
`sort_values(by="Time step")`, which uses pandas' default `kind="quicksort"` -- **not a stable
sort**. Rows sharing the same `Time step` can end up in a different relative order depending on
environment/pandas version, which feeds into `RandomForestClassifier`'s bootstrap sampling.
Confirmed empirically: an independent re-run of that notebook's exact RF cell (same code, same
data, same `random_state=42`) produced a different confusion matrix and a different top-20
feature-importance ranking than what's saved there, traced entirely to this sort instability.

Here we sort by `[Time step, txId]` with `kind="mergesort"` (stable) instead, so results in
*this* notebook are reproducible run to run. **This means the RF trained below will not
bit-for-bit match `02_Data_Processing.ipynb`'s Model 2 baseline** -- that's expected and fine,
since this model exists only to rank features for audit triage, not as a reported result. The
official Model 1/2 numbers remain the ones in `02_Data_Processing.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"

tx_features = pd.read_csv(DATA_DIR + r"\txs_features.csv")
tx_classes = pd.read_csv(DATA_DIR + r"\txs_classes.csv")
tx_data = tx_features.merge(tx_classes, on="txId", how="inner")

print("tx_features shape:", tx_features.shape)
print("tx_classes shape :", tx_classes.shape)
print("tx_data shape    :", tx_data.shape)

tx_features shape: (203769, 184)
tx_classes shape : (203769, 2)
tx_data shape    : (203769, 185)


In [3]:
augmented_features = [
    "in_txs_degree", "out_txs_degree", "total_BTC", "fees", "size",
    "num_input_addresses", "num_output_addresses",
    "in_BTC_min", "in_BTC_max", "in_BTC_mean", "in_BTC_median", "in_BTC_total",
    "out_BTC_min", "out_BTC_max", "out_BTC_mean", "out_BTC_median", "out_BTC_total",
]

labelled_data = tx_data[tx_data["class"].isin([1, 2])].copy()
labelled_data["target"] = (labelled_data["class"] == 1).astype(int)

print("Labelled dataset shape:", labelled_data.shape)
print("\nTarget distribution:")
print(labelled_data["target"].value_counts().sort_index())

Labelled dataset shape: (46564, 186)

Target distribution:
target
0    42019
1     4545
Name: count, dtype: int64


In [4]:
model_feature_cols = [c for c in tx_features.columns if c != "txId"]

# Deterministic tiebreak (see note above): stable mergesort on [Time step, txId].
labelled_data = labelled_data.sort_values(
    by=["Time step", "txId"], kind="mergesort"
).reset_index(drop=True)

unique_time_steps = sorted(labelled_data["Time step"].unique())
n_steps = len(unique_time_steps)
train_end = int(n_steps * 0.70)
val_end = int(n_steps * 0.85)
train_steps = unique_time_steps[:train_end]
val_steps = unique_time_steps[train_end:val_end]
test_steps = unique_time_steps[val_end:]

train_data = labelled_data[labelled_data["Time step"].isin(train_steps)].copy()
val_data = labelled_data[labelled_data["Time step"].isin(val_steps)].copy()
test_data = labelled_data[labelled_data["Time step"].isin(test_steps)].copy()

print(f"Train time steps: {train_steps[0]} to {train_steps[-1]} -> {len(train_data)} rows")
print(f"Validation time steps: {val_steps[0]} to {val_steps[-1]} -> {len(val_data)} rows")
print(f"Test time steps: {test_steps[0]} to {test_steps[-1]} -> {len(test_data)} rows")

Train time steps: 1 to 34 -> 29894 rows
Validation time steps: 35 to 41 -> 7829 rows
Test time steps: 42 to 49 -> 8841 rows


In [5]:
X_train = train_data[model_feature_cols].copy()
y_train = train_data["target"].copy()
X_val = val_data[model_feature_cols].copy()
y_val = val_data["target"].copy()
X_test = test_data[model_feature_cols].copy()
y_test = test_data["target"].copy()

for data in [X_train, X_val, X_test]:
    data["augmented_features_missing"] = data[augmented_features].isna().all(axis=1).astype(int)

# Medians computed from training data only.
augmented_medians = X_train[augmented_features].median()
X_train[augmented_features] = X_train[augmented_features].fillna(augmented_medians)
X_val[augmented_features] = X_val[augmented_features].fillna(augmented_medians)
X_test[augmented_features] = X_test[augmented_features].fillna(augmented_medians)

final_feature_cols = model_feature_cols + ["augmented_features_missing"]
X_train_rf = X_train[final_feature_cols]
X_val_rf = X_val[final_feature_cols]
X_test_rf = X_test[final_feature_cols]

print("Final feature count:", len(final_feature_cols))
print("X_train_rf:", X_train_rf.shape, " X_val_rf:", X_val_rf.shape, " X_test_rf:", X_test_rf.shape)

Final feature count: 184
X_train_rf: (29894, 184)  X_val_rf: (7829, 184)  X_test_rf: (8841, 184)


In [6]:
rf_model = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)
rf_model.fit(X_train_rf, y_train)

y_test_prob = rf_model.predict_proba(X_test_rf)[:, 1]
y_test_pred = (y_test_prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print("Reference-model TEST confusion matrix (threshold 0.5):")
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print("\n(Reference model for audit triage only -- expected to differ slightly")
print("from 02_Data_Processing.ipynb's official Model 2 baseline; see the note above.)")

Reference-model TEST confusion matrix (threshold 0.5):
TN=8401 FP=32 FN=216 TP=192

(Reference model for audit triage only -- expected to differ slightly
from 02_Data_Processing.ipynb's official Model 2 baseline; see the note above.)


In [7]:
rf_importance = pd.DataFrame({
    "Feature": final_feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print("Top 20 features (this reference model):")
display(rf_importance.head(20))

Top 20 features (this reference model):


,Feature,Importance
0,Local_feature_53,0.058381
1,Local_feature_55,0.048364
2,Local_feature_41,0.046895
3,size,0.041458
4,Local_feature_43,0.037388
5,Local_feature_49,0.036475
6,Local_feature_47,0.029423
7,num_output_addresses,0.025601
8,Local_feature_5,0.024548
9,Local_feature_90,0.023804


## Stage 0: feature inventory

Classify all 166 inherited features by documented/pattern-inferred/unverified semantics.
Per project rule: no invented meanings for the "not verified" bucket.

In [8]:
feature_names = (
    ["Time step"]
    + [f"Local_feature_{i}" for i in range(1, 94)]
    + [f"Aggregate_feature_{i}" for i in range(1, 73)]
)
assert len(feature_names) == 166

documented_meanings = {
    "Local_feature_53": "Minimum lifetime of input addresses",
    "Local_feature_55": "Standard deviation of lifetime of input addresses",
    "Local_feature_41": "Minimum incoming transactions of input addresses",
    "Local_feature_43": "Standard deviation of incoming transactions of input addresses",
    "Local_feature_47": "Minimum outgoing transactions of input addresses",
    "Local_feature_49": "Standard deviation of outgoing transactions of input addresses",
    "Local_feature_90": "Mean lifetime of output addresses",
    "Local_feature_5":  "Number of transaction outputs",
    "Local_feature_14": "Spearman correlation involving input values and indices",
    "Local_feature_18": "Maximum output value",
    "Local_feature_23": "Minimum received satoshi of input addresses",
    "Local_feature_25": "Standard deviation of received satoshi of input addresses",
    "Local_feature_65": "Minimum BTC sent by output addresses",
}

pattern_inferred_meanings = {
    "Local_feature_54": "Maximum lifetime of input addresses (pattern-inferred sibling of 53/55)",
    "Local_feature_42": "Maximum incoming transactions of input addresses (pattern-inferred sibling of 41/43)",
    "Local_feature_48": "Maximum outgoing transactions of input addresses (pattern-inferred sibling of 47/49)",
    "Local_feature_24": "Maximum received satoshi of input addresses (pattern-inferred sibling of 23/25)",
    "Local_feature_29": "Minimum BTC sent by input addresses (pattern-inferred, per recovered-dictionary note that 29+ concern input-address BTC sent)",
    "Local_feature_30": "Maximum BTC sent by input addresses (pattern-inferred sibling of 29)",
}

rows = []
for i, name in enumerate(feature_names, start=1):
    if name in documented_meanings:
        source, meaning = "Documented", documented_meanings[name]
    elif name in pattern_inferred_meanings:
        source, meaning = "Pattern-inferred", pattern_inferred_meanings[name]
    else:
        source, meaning = "Not verified", "Semantic mapping not verified"
    rows.append({
        "Feature_Number": i,
        "Feature": name,
        "Semantic_Source": source,
        "Documented_Meaning": meaning,
    })

feature_audit = pd.DataFrame(rows)

print("Total features catalogued:", len(feature_audit))
print(feature_audit["Semantic_Source"].value_counts())
display(feature_audit[feature_audit["Semantic_Source"] != "Not verified"])

Total features catalogued: 166
Semantic_Source
Not verified        147
Documented           13
Pattern-inferred      6
Name: count, dtype: int64


,Feature_Number,Feature,Semantic_Source,Documented_Meaning
5,6,Local_feature_5,Documented,Number of transaction outputs
14,15,Local_feature_14,Documented,Spearman correlation involving input values an...
18,19,Local_feature_18,Documented,Maximum output value
23,24,Local_feature_23,Documented,Minimum received satoshi of input addresses
24,25,Local_feature_24,Pattern-inferred,Maximum received satoshi of input addresses (p...
25,26,Local_feature_25,Documented,Standard deviation of received satoshi of inpu...
29,30,Local_feature_29,Pattern-inferred,Minimum BTC sent by input addresses (pattern-i...
30,31,Local_feature_30,Pattern-inferred,Maximum BTC sent by input addresses (pattern-i...
41,42,Local_feature_41,Documented,Minimum incoming transactions of input addresses
42,43,Local_feature_42,Pattern-inferred,Maximum incoming transactions of input address...


## Stage 1: triage -- merge in RF importance, flag audit priority

High priority = documented/pattern-inferred semantics **and** top-30 RF importance.
This is the candidate list for Stage 2 (staticness) and Stage 3 (reconstruction) -- not results yet.

In [9]:
feature_audit = feature_audit.merge(rf_importance, on="Feature", how="left")
feature_audit["RF_Importance_Rank"] = feature_audit["Importance"].rank(ascending=False, method="min")

audit_priority = (
    (feature_audit["Semantic_Source"] != "Not verified")
    & (feature_audit["RF_Importance_Rank"] <= 30)
)
feature_audit["Audit_Priority"] = np.where(audit_priority, "High", "Low")

print("High-priority features (documented/pattern-inferred AND top-30 RF importance):")
display(
    feature_audit[feature_audit["Audit_Priority"] == "High"]
    .sort_values("RF_Importance_Rank")
    [["Feature", "Semantic_Source", "Documented_Meaning", "Importance", "RF_Importance_Rank"]]
)

High-priority features (documented/pattern-inferred AND top-30 RF importance):


,Feature,Semantic_Source,Documented_Meaning,Importance,RF_Importance_Rank
53,Local_feature_53,Documented,Minimum lifetime of input addresses,0.058381,1.0
55,Local_feature_55,Documented,Standard deviation of lifetime of input addresses,0.048364,2.0
41,Local_feature_41,Documented,Minimum incoming transactions of input addresses,0.046895,3.0
43,Local_feature_43,Documented,Standard deviation of incoming transactions of...,0.037388,4.0
49,Local_feature_49,Documented,Standard deviation of outgoing transactions of...,0.036475,5.0
47,Local_feature_47,Documented,Minimum outgoing transactions of input addresses,0.029423,6.0
5,Local_feature_5,Documented,Number of transaction outputs,0.024548,7.0
90,Local_feature_90,Documented,Mean lifetime of output addresses,0.023804,8.0
14,Local_feature_14,Documented,Spearman correlation involving input values an...,0.020184,9.0
18,Local_feature_18,Documented,Maximum output value,0.017759,10.0
